
# Clasificación No Lineal con SVM



Aunque los clasificadores SVM lineales son eficientes y funcionan sorprendentemente bien en muchos casos, muchos conjuntos de datos ni siquiera se aproximan a ser linealmente separables. Un enfoque para manejar conjuntos de datos no lineales es añadir más características, como características polinómicas (como hiciste en el Capítulo 4); en algunos casos, esto puede dar como resultado un conjunto de datos linealmente separable.

Considera el gráfico de la izquierda en la Figura 5-5: representa un conjunto de datos simple con una sola característica, \(x_1\). Como puedes ver, este conjunto de datos no es linealmente separable. Pero si añades una segunda característica \(x_2 = (x_1)^2\), el conjunto de datos 2D resultante es perfectamente linealmente separable.

**Figura 5-5. Adición de características para hacer un conjunto de datos linealmente separable**

```python
# Código para generar la Figura 5-5
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_circles

# Generar datos no lineales (círculos concéntricos)
X, y = make_circles(n_samples=100, factor=0.5, noise=0.05, random_state=42)

plt.figure(figsize=(8, 4))
plt.subplot(121)
plt.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.Paired)
plt.xlabel("$x_1$")
plt.ylabel("$x_2$")
plt.title("Datos originales (no separables)")

# Añadir la característica polinómica x2 = x1^2
X_new = np.c_[X, X[:, 0]**2]  # Añadir x1^2 como tercera dimensión
plt.subplot(122, projection='3d')
plt.scatter(X_new[:, 0], X_new[:, 1], X_new[:, 2], c=y, cmap=plt.cm.Paired)
plt.xlabel("$x_1$")
plt.ylabel("$x_2$")
plt.zlabel("$x_1^2$")
plt.title("Datos transformados (separables)")
plt.tight_layout()
plt.show()
```

Para implementar esta idea usando Scikit-Learn, crea un `Pipeline` que contenga un transformador `PolynomialFeatures` (discutido en "Regresión Polinómica"), seguido de un `StandardScaler` y un `LinearSVC`. Probemos esto en el conjunto de datos `moons`: es un conjunto de datos de juguete para clasificación binaria en el que los puntos de datos tienen forma de dos semicírculos entrelazados (ver Figura 5-6). Puedes generar este conjunto de datos usando la función `make_moons()`:

```python
from sklearn.datasets import make_moons
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.svm import LinearSVC

X, y = make_moons(n_samples=100, noise=0.15, random_state=42)

polynomial_svm_clf = Pipeline([
    ("poly_features", PolynomialFeatures(degree=3)),
    ("scaler", StandardScaler()),
    ("svm_clf", LinearSVC(C=10, loss="hinge", random_state=42, dual=False))
])

polynomial_svm_clf.fit(X, y)
```

**Figura 5-6. Clasificador SVM lineal usando características polinómicas**

```python
# Código para generar la Figura 5-6
# Función de ayuda para graficar límites de decisión (adaptada del código anterior)
def plot_predictions(clf, axes):
    x0s = np.linspace(axes[0], axes[1], 100)
    x1s = np.linspace(axes[2], axes[3], 100)
    x0, x1 = np.meshgrid(x0s, x1s)
    X = np.c_[x0.ravel(), x1.ravel()]
    y_pred = clf.predict(X).reshape(x0.shape)
    plt.contourf(x0, x1, y_pred, alpha=0.2, cmap=plt.cm.Paired)

plt.figure(figsize=(6, 4))
plot_predictions(polynomial_svm_clf, [-1.5, 2.5, -1, 1.5])
plt.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.Paired, s=50)
plt.xlabel("$x_1$")
plt.ylabel("$x_2$")
plt.title("SVM con características polinómicas (grado 3)")
plt.show()
```

## Kernel Polinómico

Agregar características polinómicas es simple de implementar y puede funcionar muy bien con todo tipo de algoritmos de Aprendizaje Automático (no solo SVM). Dicho esto, con un grado polinómico bajo, este método no puede manejar conjuntos de datos muy complejos, y con un grado polinómico alto, crea una gran cantidad de características, lo que hace que el modelo sea demasiado lento.

Afortunadamente, cuando se usan SVM, se puede aplicar una técnica matemática casi milagrosa llamada **el truco del kernel** (explicado en breve). El truco del kernel permite obtener el mismo resultado que si hubieras agregado muchas características polinómicas, incluso con polinomios de grado muy alto, sin tener que añadirlas realmente. Por lo tanto, no hay una explosión combinatoria en el número de características porque en realidad no agregas ninguna. Este truco está implementado por la clase `SVC`. Probémoslo en el conjunto de datos `moons`:

```python
from sklearn.svm import SVC

poly_kernel_svm_clf = Pipeline([
    ("scaler", StandardScaler()),
    ("svm_clf", SVC(kernel="poly", degree=3, coef0=1, C=5))
])

poly_kernel_svm_clf.fit(X, y)
```

Este código entrena un clasificador SVM usando un kernel polinómico de tercer grado. Se representa en la parte izquierda de la Figura 5-7. A la derecha se muestra otro clasificador SVM que utiliza un kernel polinómico de grado 10. Obviamente, si tu modelo está sobreajustando, es posible que desees reducir el grado del polinomio. Por el contrario, si está subajustando, puedes intentar aumentarlo. El hiperparámetro `coef0` controla cuánto influyen en el modelo los polinomios de alto grado en comparación con los de bajo grado.

**Figura 5-7. Clasificadores SVM con kernel polinómico**

```python
# Código para generar la Figura 5-7
plt.figure(figsize=(10, 4))

for idx, (degree, C) in enumerate([(3, 5), (10, 1)]):
    plt.subplot(1, 2, idx+1)
    poly_kernel_svm_clf = Pipeline([
        ("scaler", StandardScaler()),
        ("svm_clf", SVC(kernel="poly", degree=degree, coef0=1, C=C))
    ])
    poly_kernel_svm_clf.fit(X, y)
    plot_predictions(poly_kernel_svm_clf, [-1.5, 2.5, -1, 1.5])
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.Paired, s=50)
    plt.xlabel("$x_1$")
    plt.ylabel("$x_2$")
    plt.title(f"Kernel polinómico (grado {degree})")

plt.tight_layout()
plt.show()
```

**CONSEJO**
Un enfoque común para encontrar los valores correctos de los hiperparámetros es usar búsqueda en cuadrícula (*grid search*, ver Capítulo 2). A menudo es más rápido realizar primero una búsqueda en cuadrícula muy gruesa y luego una búsqueda más fina alrededor de los mejores valores encontrados. Tener una buena idea de lo que realmente hace cada hiperparámetro también puede ayudarte a buscar en la parte correcta del espacio de hiperparámetros.

## Características de Similitud

Otra técnica para abordar problemas no lineales es añadir características calculadas usando una **función de similitud**, que mide cuánto se parece cada instancia a un **punto de referencia** (*landmark*) particular. Por ejemplo, tomemos el conjunto de datos 1D discutido anteriormente y agreguemos dos puntos de referencia en \(x = -2\) y \(x = 1\) (ver el gráfico de la izquierda en la Figura 5-8). A continuación, definamos la función de similitud como la **Función de Base Radial Gaussiana (RBF)** con \(\gamma = 0.3\) (ver Ecuación 5-1).

**Ecuación 5-1. RBF Gaussiana**
\[
\phi_{\gamma}(x, \ell) = \exp(-\gamma \|x - \ell\|^2)
\]

Esta es una función con forma de campana que varía de 0 (muy lejos del punto de referencia) a 1 (en el punto de referencia). Ahora estamos listos para calcular las nuevas características. Por ejemplo, consideremos la instancia \(x = -1\): está ubicada a una distancia de 1 del primer punto de referencia y 2 del segundo. Por lo tanto, sus nuevas características son \(x_2 = \exp(-0.3 \times 1^2) \approx 0.74\) y \(x_3 = \exp(-0.3 \times 2^2) \approx 0.30\). El gráfico de la derecha en la Figura 5-8 muestra el conjunto de datos transformado (eliminando las características originales). Como puedes ver, ahora es linealmente separable.

**Figura 5-8. Características de similitud usando la RBF Gaussiana**

```python
# Código para generar la Figura 5-8
def gaussian_rbf(x, landmark, gamma):
    return np.exp(-gamma * (x - landmark)**2)

# Datos 1D de ejemplo
X_1d = np.linspace(-4, 4, 100).reshape(-1, 1)
landmarks = np.array([-2, 1])
gamma = 0.3

# Calcular características de similitud
X_sim = np.c_[gaussian_rbf(X_1d, landmarks[0], gamma),
              gaussian_rbf(X_1d, landmarks[1], gamma)]

plt.figure(figsize=(8, 4))
plt.subplot(121)
plt.scatter(X_1d, np.zeros_like(X_1d), marker='x', color='k')
plt.axvline(x=landmarks[0], color='r', linestyle='--', label='Punto de referencia 1')
plt.axvline(x=landmarks[1], color='b', linestyle='--', label='Punto de referencia 2')
plt.xlabel("$x_1$")
plt.ylabel("Densidad")
plt.title("Datos originales con puntos de referencia")
plt.legend()

plt.subplot(122)
plt.scatter(X_sim[:, 0], X_sim[:, 1], c=np.zeros_like(X_1d).ravel(), cmap=plt.cm.Paired)
plt.xlabel("$x_2$")
plt.ylabel("$x_3$")
plt.title("Datos transformados (separables)")
plt.tight_layout()
plt.show()
```

Puede que te preguntes cómo seleccionar los puntos de referencia. El enfoque más simple es crear un punto de referencia en la ubicación de cada instancia en el conjunto de datos. Esto crea muchas dimensiones y, por lo tanto, aumenta las posibilidades de que el conjunto de entrenamiento transformado sea linealmente separable. La desventaja es que un conjunto de entrenamiento con \(m\) instancias y \(n\) características se transforma en un conjunto de entrenamiento con \(m\) instancias y \(m\) características (asumiendo que eliminas las características originales). Si tu conjunto de entrenamiento es muy grande, terminas con un número igualmente grande de características.

## Kernel RBF Gaussiano

Al igual que el método de características polinómicas, el método de características de similitud puede ser útil con cualquier algoritmo de Aprendizaje Automático, pero puede ser computacionalmente costoso calcular todas las características adicionales, especialmente en conjuntos de entrenamiento grandes. Una vez más, el truco del kernel hace su magia en SVM, permitiendo obtener un resultado similar al que se obtendría si se hubieran añadido muchas características de similitud. Probemos la clase `SVC` con el kernel RBF Gaussiano:

```python
rbf_kernel_svm_clf = Pipeline([
    ("scaler", StandardScaler()),
    ("svm_clf", SVC(kernel="rbf", gamma=5, C=0.001))
])

rbf_kernel_svm_clf.fit(X, y)
```

Este modelo se representa en la parte inferior izquierda de la Figura 5-9. Los otros gráficos muestran modelos entrenados con diferentes valores de los hiperparámetros \(\gamma\) (gamma) y \(C\). Aumentar \(\gamma\) hace que la curva con forma de campana sea más estrecha (ver los gráficos de la derecha en la Figura 5-8). Como resultado, el rango de influencia de cada instancia es más pequeño: el límite de decisión termina siendo más irregular, serpenteando alrededor de instancias individuales. Por el contrario, un valor pequeño de \(\gamma\) hace que la curva con forma de campana sea más ancha: las instancias tienen un rango de influencia mayor y el límite de decisión termina siendo más suave. Por lo tanto, \(\gamma\) actúa como un hiperparámetro de regularización: si tu modelo está sobreajustando, deberías reducirlo; si está subajustando, deberías aumentarlo (similar al hiperparámetro \(C\)).

**Figura 5-9. Clasificadores SVM usando un kernel RBF**

```python
# Código para generar la Figura 5-9
gamma_values = [0.1, 5]
C_values = [0.001, 1000]

plt.figure(figsize=(12, 6))
for i, gamma in enumerate(gamma_values):
    for j, C in enumerate(C_values):
        plt.subplot(2, 2, i*2 + j + 1)
        rbf_kernel_svm_clf = Pipeline([
            ("scaler", StandardScaler()),
            ("svm_clf", SVC(kernel="rbf", gamma=gamma, C=C))
        ])
        rbf_kernel_svm_clf.fit(X, y)
        plot_predictions(rbf_kernel_svm_clf, [-1.5, 2.5, -1, 1.5])
        plt.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.Paired, s=50)
        plt.xlabel("$x_1$")
        plt.ylabel("$x_2$")
        plt.title(f"RBF, $\gamma={gamma}$, C={C}")

plt.tight_layout()
plt.show()
```

Existen otros kernels, pero se utilizan con mucha menos frecuencia. Algunos kernels están especializados para estructuras de datos específicas. Los **kernels de cadenas** (*string kernels*) se utilizan a veces para clasificar documentos de texto o secuencias de ADN (por ejemplo, usando el kernel de subsecuencias de cadenas o kernels basados en la distancia de Levenshtein).

**CONSEJO**
Con tantos kernels para elegir, ¿cómo puedes decidir cuál usar? Como regla general, siempre debes probar primero el kernel lineal (recuerda que `LinearSVC` es mucho más rápido que `SVC(kernel="linear")`), especialmente si el conjunto de entrenamiento es muy grande o tiene muchas características. Si el conjunto de entrenamiento no es demasiado grande, también deberías probar el kernel RBF Gaussiano; funciona bien en la mayoría de los casos. Luego, si tienes tiempo y potencia de cálculo, puedes experimentar con algunos otros kernels, usando validación cruzada y búsqueda en cuadrícula. Te convendría experimentar así especialmente si hay kernels especializados para la estructura de datos de tu conjunto de entrenamiento.

## Complejidad Computacional

La clase `LinearSVC` se basa en la biblioteca **liblinear**, que implementa un algoritmo optimizado para SVM lineales. No soporta el truco del kernel, pero escala casi linealmente con el número de instancias de entrenamiento y el número de características. Su complejidad temporal de entrenamiento es aproximadamente \(O(m \times n)\). El algoritmo tarda más si se requiere una precisión muy alta. Esto está controlado por el hiperparámetro de tolerancia \(\epsilon\) (llamado `tol` en Scikit-Learn). En la mayoría de las tareas de clasificación, la tolerancia predeterminada es suficiente.

La clase `SVC` se basa en la biblioteca **libsvm**, que implementa un algoritmo que soporta el truco del kernel. La complejidad temporal de entrenamiento suele estar entre \(O(m \times n)\) y \(O(m^2 \times n)\). Desafortunadamente, esto significa que se vuelve terriblemente lento cuando el número de instancias de entrenamiento es grande (por ejemplo, cientos de miles de instancias). Este algoritmo es perfecto para conjuntos de entrenamiento complejos, pequeños o medianos. Escala bien con el número de características, especialmente con características dispersas (es decir, cuando cada instancia tiene pocas características no nulas). En este caso, el algoritmo escala aproximadamente con el número promedio de características no nulas por instancia.

La Tabla 5-1 compara las clases de clasificación SVM de Scikit-Learn.

**Tabla 5-1. Comparación de las clases de Scikit-Learn para clasificación SVM**

| Clase | Complejidad temporal | Soporte *out-of-core* | Escalado requerido | Truco del kernel |
| :--- | :--- | :--- | :--- | :--- |
| `LinearSVC` | \(O(m \times n)\) | No | Sí | No |
| `SVC` (kernel lineal) | \(O(m \times n)\) a \(O(m^2 \times n)\) | No | Sí | Sí |
| `SVC` (otros kernels) | \(O(m^2 \times n)\) a \(O(m^3 \times n)\) | No | Sí | Sí |
| `SGDClassifier` | \(O(m \times n)\) | Sí | Sí | No |